### Bronze → Silver: decode GTFS-Realtime vehicle positions into typed vehicle-level rows

- **Decode:** `from_protobuf` with the shared compiled descriptor (`gtfs_realtime.desc`) — the **same**
  `FeedMessage` proto as trip updates, decoded JVM-side (no Python bindings on executors).
- **Grain:** one row per vehicle per feed snapshot (explode `entity[]` → `vehicle`).
- **Incremental:** streaming read of bronze + checkpoint → each run decodes only newly-appended rows.

> **DRAFT — written against the GTFS-RT spec, not yet validated against real V/Line bytes.** Which optional
> fields V/Line actually populates (`license_plate`, `bearing`, `occupancy_status`, `current_status`, and
> whether `vehicle.trip` is present) needs confirming once the feed is flowing. Run as a serverless
> **job** — Structured Streaming isn't supported over Spark Connect.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.protobuf.functions import from_protobuf

# Catalog is passed as a job parameter (default -> ${var.catalog} per target); the default here
# keeps interactive runs working. Schemas/volumes are the same across dev and prod.
dbutils.widgets.text("catalog", "transport_vic_dev")
CATALOG = dbutils.widgets.get("catalog")

BRONZE_TBL = f"{CATALOG}.`01_bronze`.vline_vehicle_positions"
SILVER_TBL = f"{CATALOG}.`02_silver`.vehicle_positions"
CHECKPOINT = f"/Volumes/{CATALOG}/02_silver/_checkpoints/vehicle_positions"
DESC_PATH  = f"/Volumes/{CATALOG}/02_silver/artifacts/gtfs_realtime.desc"
MESSAGE    = "transit_realtime.FeedMessage"

# Shared with the trip-updates silver — same proto, same descriptor volume.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.`02_silver`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.`02_silver`._checkpoints")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.`02_silver`.artifacts")

In [ ]:
# Build the FileDescriptorSet ONCE from the installed bindings and cache it in the Volume — the SAME
# .desc the trip-updates silver uses (one FeedMessage proto covers both feeds). Skipped once it exists.
# Needs gtfs-realtime-bindings in the job environment; after the .desc exists it is never imported again.
import os

if not os.path.exists(DESC_PATH):
    from google.protobuf import descriptor_pb2
    from google.transit import gtfs_realtime_pb2

    fds, seen = descriptor_pb2.FileDescriptorSet(), set()
    def _add(fd):
        if fd.name in seen:
            return
        seen.add(fd.name)
        for dep in fd.dependencies:
            _add(dep)
        fd.CopyToProto(fds.file.add())

    _add(gtfs_realtime_pb2.DESCRIPTOR)
    with open(DESC_PATH, "wb") as f:
        f.write(fds.SerializeToString())
    print("wrote descriptor ->", DESC_PATH)
else:
    print("descriptor already present ->", DESC_PATH)

In [ ]:
# Pre-create silver with explicit types. lat/lon are the point of this feed, but position is technically
# optional in the proto, so keep them nullable. license_plate/occupancy_status/congestion_level dropped
# — V/Line never populates them (all counted 0). current_status kept raw-nullable.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_TBL} (
  feed_ts           TIMESTAMP NOT NULL,
  entity_id         STRING    NOT NULL,
  trip_id           STRING,
  route_id          STRING,
  direction_id      INT,
  start_date        DATE,
  start_time        STRING,               -- GTFS allows >24:00:00
  trip_rel          STRING,
  vehicle_id        STRING,
  vehicle_label     STRING,
  latitude          DOUBLE,
  longitude         DOUBLE,
  bearing           DOUBLE,
  speed             DOUBLE,
  current_stop_seq  INT,
  stop_id           STRING,
  current_status    STRING,               -- INCOMING_AT / STOPPED_AT / IN_TRANSIT_TO (null if unset)
  vehicle_ts        TIMESTAMP,            -- when THIS vehicle's position was recorded
  event_time        TIMESTAMP,            -- coalesce(vehicle_ts, feed_ts) for sorting/clustering
  source_file       STRING    NOT NULL,
  _ingest_ts        TIMESTAMP,
  _silver_ts        TIMESTAMP
)
USING DELTA
CLUSTER BY (start_date, route_id)
""")

In [ ]:
# Streaming decode. from_protobuf yields a typed struct: position lat/lon/bearing/speed are FLOAT,
# timestamps are LONG epoch seconds (cast to timestamp), enums are their string names.
bronze = (spark.readStream
          .option("skipChangeCommits", "true")   # tolerate future OPTIMIZE on bronze
          .table(BRONZE_TBL))

decoded = bronze.select(
    F.col("path").alias("source_file"),
    F.col("_ingest_ts"),
    from_protobuf(F.col("content"), MESSAGE, descFilePath=DESC_PATH).alias("feed"),
)

# Explode entities and keep only vehicle-position ones (a pure VP feed = all of them; filter is defensive).
entities = (decoded
    .withColumn("feed_ts", F.col("feed.header.timestamp").cast("timestamp"))
    .select("source_file", "_ingest_ts", "feed_ts", F.explode("feed.entity").alias("e"))
    .filter(F.col("e.vehicle").isNotNull()))

vehicles = (entities.select(
        "feed_ts",
        F.col("e.id").alias("entity_id"),
        F.col("e.vehicle.trip.trip_id").alias("trip_id"),
        F.col("e.vehicle.trip.route_id").alias("route_id"),
        F.col("e.vehicle.trip.direction_id").cast("int").alias("direction_id"),
        F.to_date("e.vehicle.trip.start_date", "yyyyMMdd").alias("start_date"),
        F.col("e.vehicle.trip.start_time").alias("start_time"),
        F.coalesce(F.col("e.vehicle.trip.schedule_relationship"), F.lit("SCHEDULED")).alias("trip_rel"),
        F.col("e.vehicle.vehicle.id").alias("vehicle_id"),
        F.col("e.vehicle.vehicle.label").alias("vehicle_label"),
        F.col("e.vehicle.position.latitude").cast("double").alias("latitude"),
        F.col("e.vehicle.position.longitude").cast("double").alias("longitude"),
        F.col("e.vehicle.position.bearing").cast("double").alias("bearing"),
        F.col("e.vehicle.position.speed").cast("double").alias("speed"),
        F.col("e.vehicle.current_stop_sequence").cast("int").alias("current_stop_seq"),
        F.col("e.vehicle.stop_id").alias("stop_id"),
        F.col("e.vehicle.current_status").alias("current_status"),      # raw-nullable: null = V/Line didn't send it
        F.col("e.vehicle.timestamp").cast("timestamp").alias("vehicle_ts"),
        "source_file", "_ingest_ts",
    )
    .withColumn("event_time", F.coalesce("vehicle_ts", "feed_ts"))
    .withColumn("_silver_ts", F.current_timestamp())
    .select(  # match the table DDL column order exactly
        "feed_ts", "entity_id", "trip_id", "route_id", "direction_id", "start_date", "start_time",
        "trip_rel", "vehicle_id", "vehicle_label", "latitude", "longitude",
        "bearing", "speed", "current_stop_seq", "stop_id", "current_status",
        "vehicle_ts", "event_time", "source_file", "_ingest_ts", "_silver_ts",
    ))

In [ ]:
# Append new rows, then stop. Separate checkpoint per query (one per streaming query).
q = (vehicles.writeStream
     .option("checkpointLocation", CHECKPOINT)
     .trigger(availableNow=True)
     .toTable(SILVER_TBL))
q.awaitTermination()
print("silver rows:", spark.table(SILVER_TBL).count())

#### Notes
- **Natural grain / key:** `(feed_ts, entity_id)` — one row per vehicle per snapshot. If the feed is
  `FULL_DATASET`, each ~2-min poll re-publishes every vehicle; silver keeps that as a position
  time-series (the vehicle's trail). Any "latest position per vehicle" collapse belongs in **gold**.
- **Dedup:** the CLAUDE.md plan is to dedup on `header.timestamp`. Landing every snapshot in bronze and
  keeping them here is deliberate — collapse/dedup downstream. If storage ever bites, dedup on `feed_ts`.
- **Append is idempotent** via the checkpoint (each bronze row decoded once) — no MERGE needed.
- **Pending validation (needs real bytes):** confirm the `current_status` default, whether
  `license_plate` / `bearing` / `occupancy_status` are populated by V/Line, and that `vehicle.trip`
  is present (it's the join key to routes/trips for the gold performance work).
- Mirrors `02_bronze_to_silver.ipynb` (trip updates); if `descFilePath` on a Volume misbehaves, read the
  bytes and pass `binaryDescriptorSet=` instead.